# Aula 1, O que é IA

Notebook executável que acompanha a aula [01-o-que-e-ia.md](../../lessons/modulo-01-introducao-ia/01-o-que-e-ia.md).

## O que vamos fazer aqui

A pergunta de fundo deste notebook é simples: como um computador descobre de que
assunto trata a pergunta de um aluno? Vamos resolver essa mesma tarefa de três
maneiras, uma para cada grande abordagem de Inteligência Artificial.

1. **IA simbólica**, em que escrevemos regras à mão.
2. **IA estatística**, em que o computador aprende com exemplos.
3. **IA generativa**, em que um modelo escreve uma resposta nova.

Comparar as três lado a lado deixa visível o que muda de uma para a outra. Os
exemplos rodam com modelos locais via Ollama, e a parte generativa funciona mesmo
sem o Ollama instalado, exibindo um aviso amigável no lugar.

## Abordagem 1, IA simbólica com regras escritas à mão

Na IA simbólica, o conhecimento é escrito de forma explícita por uma pessoa. Aqui,
montamos um dicionário que liga palavras-chave a temas e classificamos a pergunta
procurando essas palavras. Repare que a máquina não aprende nada, ela apenas segue
as regras que demos.

In [ ]:
# O conhecimento é escrito explicitamente por uma pessoa, na forma de regras.
REGRAS = {
    'derivada': 'cálculo',
    'integral': 'cálculo',
    'limite': 'cálculo',
    'vetor': 'álgebra linear',
    'matriz': 'álgebra linear',
    'probabilidade': 'estatística',
    'média': 'estatística',
}


def classificar_por_regras(pergunta):
    """Classifica a pergunta procurando palavras-chave conhecidas."""
    texto = pergunta.lower()
    for palavra, tema in REGRAS.items():
        if palavra in texto:
            return tema
    return 'desconhecido'


print(classificar_por_regras('Como calculo a derivada de x ao quadrado?'))
print(classificar_por_regras('O que é desvio padrão?'))

A primeira pergunta é classificada como `cálculo`, porque contém a palavra
derivada. Já a segunda devolve `desconhecido`, embora seja claramente de
estatística, pois não existe regra para as palavras dela. Esse é o limite da
abordagem simbólica: alguém precisa prever cada caso, e cobrir tudo à mão é
inviável.

## Abordagem 2, IA estatística que aprende com exemplos

Agora invertemos a lógica. Em vez de escrever regras, fornecemos exemplos já
rotulados e deixamos o modelo descobrir os padrões sozinho. Usamos TF-IDF para
transformar o texto em números e uma regressão logística para classificar. Esta
célula usa o scikit-learn, que entra com `pip install -r requirements.txt`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Em vez de regras, fornecemos exemplos já rotulados.
perguntas = [
    'como calculo a derivada de uma função',
    'qual a integral de x dx',
    'o que é um limite no cálculo',
    'como multiplico duas matrizes',
    'o que é um vetor unitário',
    'como calculo a média de uma amostra',
    'o que significa desvio padrão',
    'como interpreto uma probabilidade',
]
temas = [
    'cálculo', 'cálculo', 'cálculo',
    'álgebra linear', 'álgebra linear',
    'estatística', 'estatística', 'estatística',
]

# O pipeline transforma o texto em números com TF-IDF e treina um classificador.
modelo = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
modelo.fit(perguntas, temas)

# Testamos com uma frase nova, que o modelo nunca viu exatamente assim.
print(modelo.predict(['como encontro o desvio padrão de uma série de notas']))

Mesmo sem uma regra para desvio padrão, o modelo classifica a frase como
estatística. Ele generaliza a partir dos exemplos parecidos que viu no treino. Essa
capacidade de acertar casos novos, e não só os memorizados, é o coração do Machine
Learning, e é exatamente o que faltava à abordagem simbólica.

## Abordagem 3, IA generativa com um modelo de linguagem local

As duas abordagens anteriores escolhem um rótulo. A IA generativa dá um passo além,
ela produz texto novo. Aqui pedimos a um modelo local, via Ollama, que escreva uma
explicação para o aluno. Se o Ollama não estiver disponível, o `try`/`except`
mostra um aviso e o notebook continua rodando.

In [ ]:
# A IA generativa não classifica apenas, ela produz texto novo.
# Usamos o Ollama, que roda um LLM localmente, sem chave de API.
try:
    import ollama

    resposta = ollama.chat(
        model='llama3.1',
        messages=[
            {
                'role': 'user',
                'content': (
                    'Explique, em duas frases e de forma simples, '
                    'o que é desvio padrão para um estudante do primeiro ano.'
                ),
            }
        ],
    )
    print(resposta['message']['content'])
except Exception as erro:
    print('Não foi possível usar o Ollama agora.')
    print('Veja docs/setup.md. Detalhe técnico:', erro)

Note o salto de capacidade. Em vez de devolver um rótulo como cálculo ou
estatística, o modelo escreveu uma explicação inteira, adaptada ao pedido. É esse
tipo de geração que dá voz aos assistentes educacionais que vamos construir ao longo
da trilha.

## Comparando as três abordagens

A célula abaixo junta tudo. Para um conjunto de perguntas de teste, ela mostra lado a
lado o tema previsto pela abordagem simbólica, o tema previsto pela abordagem
estatística e uma resposta curta gerada pela abordagem generativa. Esse comparador é
o ponto de partida do projeto da aula.

In [ ]:
def explicar_com_llm(pergunta):
    try:
        import ollama
        resposta = ollama.chat(
            model='llama3.1',
            messages=[{'role': 'user', 'content': 'Responda em uma frase: ' + pergunta}],
        )
        return resposta['message']['content'].strip()
    except Exception:
        return '(LLM indisponível, veja docs/setup.md)'


perguntas_teste = [
    'Como calculo a derivada de x ao quadrado?',
    'O que é desvio padrão?',
    'Como multiplico duas matrizes?',
    'O que é um limite?',
    'Como interpreto uma probabilidade?',
]

for p in perguntas_teste:
    simbolico = classificar_por_regras(p)
    estatistico = modelo.predict([p])[0]
    print('Pergunta:', p)
    print('  Simbólico   :', simbolico)
    print('  Estatístico :', estatistico)
    print('  Generativo  :', explicar_com_llm(p))
    print()

## Conclusão e projeto da aula

Rodando o comparador, dá para sentir as diferenças. A abordagem simbólica é
transparente, mas trava quando a palavra não está nas regras. A estatística
generaliza para frases novas. A generativa escreve respostas originais. Para o
projeto da aula, estenda este comparador com pelo menos cinco perguntas suas e
escreva um parágrafo dizendo em que situação cada abordagem se sai melhor.